# Chapter 9 — Text Processing and Multiclass Classification

**Book:** scikit-learn Cookbook, Third Edition — John Sukup (Packt Publishing, 2025)

This notebook follows the chapter sequence and demonstrates the core methods through executable examples. Numerical results appear immediately below the code, followed by a discussion of how to read them. The code examples are self-contained educational reproductions and extensions, not a word-for-word transcription of the text.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
np.set_printoptions(precision=3, suppress=True)
RANDOM_STATE=42

## 1. 1. Preparing text for machine learning

Text must be transformed into numerical features before a conventional scikit-learn estimator can use it. Simple normalization can help, but removing meaningful words may harm classification.

In [2]:
documents=['Amazing battery life and sharp display','Battery drains quickly and the screen is dim','Great camera with clear photos','Poor photos and a blurry camera','Fast performance and responsive interface','Slow interface and frustrating delays']
labels=['positive','negative','positive','negative','positive','negative']
print('Documents:',len(documents)); print('Example:',documents[0].lower())

Documents: 6
Example: amazing battery life and sharp display


**Interpretation.** The example contains positive and negative short reviews. Their wording highlights why keyword occurrences can be useful features, even though they do not capture all aspects of meaning.

## 2. 2. Count vectorization

CountVectorizer constructs a vocabulary and records term frequencies for each document. The resulting feature matrix is usually sparse.

In [3]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(stop_words='english')
mat=cv.fit_transform(documents)
print('Feature matrix shape:',mat.shape)
print('Vocabulary sample:',cv.get_feature_names_out()[:12])
print('Nonzero counts:',mat.nnz)

Feature matrix shape: (6, 22)
Vocabulary sample: ['amazing' 'battery' 'blurry' 'camera' 'clear' 'delays' 'dim' 'display'
 'drains' 'fast' 'frustrating' 'great']
Nonzero counts: 26


**Interpretation.** Each column represents a learned token and each row a document. The small number of nonzero entries illustrates the sparsity typical of text data.

## 3. 3. TF-IDF weighting

TF-IDF reduces the influence of words that appear in many documents and emphasizes terms that are more distinctive.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
v=TfidfVectorizer(stop_words='english')
X=v.fit_transform(documents)
first=X[0].toarray().ravel(); ix=first.argsort()[-5:][::-1]
print([(v.get_feature_names_out()[i],round(first[i],3)) for i in ix if first[i]>0])

[('sharp', np.float64(0.463)), ('life', np.float64(0.463)), ('display', np.float64(0.463)), ('amazing', np.float64(0.463)), ('battery', np.float64(0.379))]


**Interpretation.** Higher weights highlight terms distinctive to the first review. TF-IDF is a weighted text representation, not a direct measurement of whether a statement is positive or negative.

## 4. 4. N-grams

N-grams represent sequences of words instead of isolated tokens, allowing models to consider local phrases such as not good.

In [5]:
grams=CountVectorizer(ngram_range=(1,2)).fit(documents)
print('Vocabulary size with unigrams and bigrams:',len(grams.get_feature_names_out()))
print('Sample bigrams:',[x for x in grams.get_feature_names_out() if ' ' in x][:12])

Vocabulary size with unigrams and bigrams: 54
Sample bigrams: ['amazing battery', 'and blurry', 'and frustrating', 'and responsive', 'and sharp', 'and the', 'battery drains', 'battery life', 'blurry camera', 'camera with', 'clear photos', 'drains quickly']


**Interpretation.** Adding bigrams increases vocabulary size but can preserve information that single words miss. Larger vocabularies may require more observations to avoid overfitting.

## 5. 5. Text classification pipeline

A pipeline can vectorize training documents and train a classifier without leaking fitted vocabulary information from the test set.

In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score
texts=['excellent quality and fast service','terrible service and broken item','very happy with this product','disappointed and unhappy purchase','fantastic experience and helpful staff','waste of money bad quality','works perfectly great purchase','slow shipment awful packaging','lovely design highly recommended','poor build and faulty parts','great value useful features','not worth buying very disappointing']
y=['pos','neg']*6
text_clf=make_pipeline(TfidfVectorizer(ngram_range=(1,2)),MultinomialNB())
s=cross_val_score(text_clf,texts,y,cv=3)
print('3-fold accuracy:',s.round(3))
text_clf.fit(texts,y)
print('New text prediction:',text_clf.predict(['great helpful quality','awful poor quality']))

3-fold accuracy: [0.   0.25 0.5 ]
New text prediction: ['pos' 'neg']


**Interpretation.** The scores come from very little text and should be treated as a demonstration rather than reliable production performance. The trained pipeline applies the same vectorizer automatically during prediction.

## 6. 6. Multiclass text classification

Several topics can be classified using the same sparse-feature workflow. Class-wise reporting is important because topic frequencies may differ.

In [7]:
from sklearn.metrics import classification_report
samples=['the football team scored a goal','the tennis match was exciting','players prepared for the basketball game','the coach planned new training','the laptop uses a fast processor','new software improves security','the smartphone needs an update','the network router was replaced','the recipe uses fresh vegetables','bake bread in a hot oven','the soup has herbs and spices','the restaurant served a tasty meal']
target=['sports']*4+['technology']*4+['food']*4
from sklearn.model_selection import LeaveOneOut, cross_val_predict
pipe=make_pipeline(TfidfVectorizer(),MultinomialNB())
preds=cross_val_predict(pipe,samples,target,cv=4)
print(classification_report(target,preds,zero_division=0))

              precision    recall  f1-score   support

        food       1.00      0.25      0.40         4
      sports       0.22      0.50      0.31         4
  technology       0.00      0.00      0.00         4

    accuracy                           0.25        12
   macro avg       0.41      0.25      0.24        12
weighted avg       0.41      0.25      0.24        12



**Interpretation.** A separate precision, recall, and F1 score is provided for each topic. The toy corpus is intentionally small, so weak scores mainly demonstrate the difficulty of learning a vocabulary from limited examples.

## 7. Text normalization and tokenization

Case conversion and simple tokenization reduce superficial spelling variation but do not solve language ambiguity.

In [8]:
import re
sentences=['The models are running well!','Modeling models can be useful.','The model is NOT perfect.']
for s in sentences: print(re.findall(r"[a-z]+",s.lower()))

['the', 'models', 'are', 'running', 'well']
['modeling', 'models', 'can', 'be', 'useful']
['the', 'model', 'is', 'not', 'perfect']


**Interpretation.** Lowercasing produces more consistent tokens; punctuation is excluded. This simplistic tokenizer is only a starting point because punctuation, negation and morphology may carry meaning.

## 8. Token frequency inspection

A term-frequency table helps understand which words dominate a collection before building a predictive model.

In [9]:
from collections import Counter
texts=['great quality fast service','bad service delayed order','great experience good quality','poor quality bad delivery','fast delivery great support','delayed service poor experience']
counts=Counter(' '.join(texts).split());print(counts.most_common(12))

[('great', 3), ('quality', 3), ('service', 3), ('fast', 2), ('bad', 2), ('delayed', 2), ('experience', 2), ('poor', 2), ('delivery', 2), ('order', 1), ('good', 1), ('support', 1)]


**Interpretation.** Very frequent words can dominate bag-of-words vectors. A frequency list provides clues for vocabulary cleanup, but frequency alone does not establish predictive value.

## 9. CountVectorizer: vocabulary and sparse matrix

CountVectorizer converts text into sparse word counts. Each row represents a document, each column a vocabulary term.

In [10]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer();M=cv.fit_transform(texts)
print('Shape:',M.shape);print('Vocabulary:',cv.get_feature_names_out());print('First two documents:\n',M[:2].toarray())

Shape: (6, 12)
Vocabulary: ['bad' 'delayed' 'delivery' 'experience' 'fast' 'good' 'great' 'order'
 'poor' 'quality' 'service' 'support']
First two documents:
 [[0 0 0 0 1 0 1 0 0 1 1 0]
 [1 1 0 0 0 0 0 1 0 0 1 0]]


**Interpretation.** The feature matrix has one row per document and one column per token. Zeroes correspond to words absent from a document; this explains why sparse storage is efficient.

## 10. Stopword filtering

Removing common terms may focus the representation on more discriminative words, although careless filtering can delete useful negations.

In [11]:
v=CountVectorizer(stop_words='english');B=v.fit_transform(['this is the best product','the product is not the best','this is a new product'])
print(v.get_feature_names_out());print(B.toarray())

['best' 'new' 'product']
[[1 0 1]
 [1 0 1]
 [0 1 1]]


**Interpretation.** Stopword filtering reduces the feature space. The effect must be checked carefully, because some removed words could reverse or qualify sentiment.

## 11. Word and character n-grams

N-grams keep some local word order, making phrases and short spelling patterns available to a model.

In [12]:
for analyzer,ngram in [('word',(1,2)),('char',(3,4))]:
 vec=CountVectorizer(analyzer=analyzer,ngram_range=ngram,max_features=20)
 m=vec.fit_transform(texts)
 print(analyzer,'shape:',m.shape,'examples:',vec.get_feature_names_out()[:8])

word shape: (6, 20) examples: ['bad' 'bad delivery' 'bad service' 'delayed' 'delayed order'
 'delayed service' 'delivery' 'delivery great']
char shape: (6, 20) examples: [' de' ' del' ' qu' ' qua' ' se' ' ser' 'ali' 'alit']


**Interpretation.** Word bigrams capture adjacent words; character grams can be helpful for spelling variants. Larger n-grams increase dimensionality and may create sparsity.

## 12. TF-IDF weighting

TF-IDF downweights ubiquitous terms and can emphasize informative words within individual documents.

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
vec=TfidfVectorizer();T=vec.fit_transform(texts)
row=pd.Series(T[0].toarray().ravel(),index=vec.get_feature_names_out()).sort_values(ascending=False)
print(row.head(8).round(3))

fast          0.564
quality       0.477
great         0.477
service       0.477
experience    0.000
delivery      0.000
delayed       0.000
bad           0.000
dtype: float64


**Interpretation.** Words that occur in the document but not throughout the corpus tend to receive higher weight. These values represent relevance to the corpus, not literal probabilities.

## 13. Sparse text representations

Real text datasets can contain many distinct terms but few present terms per document.

In [14]:
print('Documents:',T.shape[0],'features:',T.shape[1]);print('Nonzeros:',T.nnz);print('Density:',round(T.nnz/(T.shape[0]*T.shape[1]),3))

Documents: 6 features: 12
Nonzeros: 24
Density: 0.333


**Interpretation.** The low density illustrates why converting large text matrices to dense arrays may consume excessive memory. Most scikit-learn text estimators accept sparse input.

## 14. Multiclass classification with pipeline

A pipeline keeps vectorization and model fitting together, preventing test text from influencing vocabulary creation.

In [15]:
from sklearn.linear_model import LogisticRegression
train_text=['red apple fruit','green apple fruit','banana sweet yellow','yellow banana sweet','blue ocean water','blue lake water']
y_train=['apple','apple','banana','banana','water','water']
clf=make_pipeline(TfidfVectorizer(),LogisticRegression(max_iter=1000)).fit(train_text,y_train)
print(clf.predict(['green apple','yellow sweet','blue water']))

['apple' 'banana' 'water']


**Interpretation.** The model estimates a category for each new sentence. Predictions here mainly demonstrate multiclass workflow; the tiny synthetic corpus does not support claims about generalization.

## 15. Multinomial Naive Bayes for documents

Multinomial Naive Bayes is a common simple baseline for word-count features because it learns term distributions for each class.

In [16]:
from sklearn.naive_bayes import MultinomialNB
bayes=make_pipeline(CountVectorizer(),MultinomialNB(alpha=1)).fit(train_text,y_train)
print('Classes:',bayes.classes_);print('Predicted classes:',bayes.predict(['sweet banana','lake water','apple fruit']))

Classes: ['apple' 'banana' 'water']
Predicted classes: ['banana' 'water' 'apple']


**Interpretation.** Naive Bayes provides a lightweight baseline and produces predictions based on word occurrence patterns. Its conditional-independence assumption is simplistic but often practical.

## 16. One-vs-rest multiclass strategy

One-vs-rest creates one binary decision problem per class. It illustrates how several binary classifiers can form a multiclass system.

In [17]:
from sklearn.multiclass import OneVsRestClassifier
ovr=make_pipeline(TfidfVectorizer(),OneVsRestClassifier(LogisticRegression(max_iter=1000))).fit(train_text,y_train)
print('OVR predictions:',ovr.predict(['green apple','blue ocean','sweet banana']))

OVR predictions: ['apple' 'water' 'banana']


**Interpretation.** The trained binary estimators compete across possible classes. The final prediction is selected from class-specific scores, not from separate unrelated decisions.

## 17. Stratified cross-validation of a text model

Text performance should be estimated using held-out documents. For this small demonstration we use a balanced synthetic collection.

In [18]:
docs=[];labels=[]
for label,words in [('sports',['team goal win','player scores goal','match team score','training game win']),('tech',['python code program','computer server cloud','software code app','network digital computer']),('food',['recipe fresh soup','bread delicious meal','cook vegetable food','fruit cake kitchen'])]:
 for s in words:docs.append(s);labels.append(label)
model=make_pipeline(TfidfVectorizer(),MultinomialNB())
print('Three-fold accuracy:',cross_val_score(model,docs,labels,cv=StratifiedKFold(3,shuffle=True,random_state=42)).round(3))

Three-fold accuracy: [0.75 1.   0.5 ]


**Interpretation.** Cross-validation separates fitting and evaluation across folds. Because this miniature corpus is constructed from distinct word lists, its scores are educational rather than evidence of a robust production classifier.

## 18. Confusion matrix for text classification

A confusion matrix shows which multiclass labels are confused with one another.

In [19]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix,classification_report
pred=cross_val_predict(model,docs,labels,cv=StratifiedKFold(3,shuffle=True,random_state=42))
print(confusion_matrix(labels,pred,labels=['sports','tech','food']))
print(classification_report(labels,pred,zero_division=0))

[[3 0 1]
 [0 4 0]
 [2 0 2]]
              precision    recall  f1-score   support

        food       0.67      0.50      0.57         4
      sports       0.60      0.75      0.67         4
        tech       1.00      1.00      1.00         4

    accuracy                           0.75        12
   macro avg       0.76      0.75      0.75        12
weighted avg       0.76      0.75      0.75        12



**Interpretation.** Diagonal cells are correct predictions and off-diagonal cells are errors. Precision and recall provide class-level detail that overall accuracy would hide.

## 19. Model coefficients as text indicators

For linear text classifiers, coefficient magnitudes describe which vocabulary features influence each class most strongly.

In [20]:
clf2=make_pipeline(TfidfVectorizer(),LogisticRegression(max_iter=1000)).fit(docs,labels)
vocab=clf2.named_steps['tfidfvectorizer'].get_feature_names_out();lr=clf2.named_steps['logisticregression']
for i,name in enumerate(lr.classes_):
 print(name,list(vocab[np.argsort(lr.coef_[i])[-4:]]))

food ['meal', 'recipe', 'soup', 'vegetable']
sports ['score', 'goal', 'team', 'win']
tech ['program', 'server', 'code', 'computer']


**Interpretation.** The selected terms reflect the artificial training corpus. Coefficients suggest word associations with classes but are not causal explanations.

## 20. Feature hashing for larger vocabularies

HashingVectorizer maps tokens into a fixed-size feature space without holding a learned vocabulary.

In [21]:
from sklearn.feature_extraction.text import HashingVectorizer
h=HashingVectorizer(n_features=32,alternate_sign=False,norm=None)
H=h.transform(texts)
print('Hashed shape:',H.shape,'nonzero features:',H.nnz)

Hashed shape: (6, 32) nonzero features: 24


**Interpretation.** Feature hashing controls matrix width and avoids a vocabulary lookup. It sacrifices easy recovery of original feature names and can create hash collisions.